# 06 — Social charts (Pillow)

Three publication-ready charts, each rendered **inline** (then saved to
`outputs/social/`). Workspace rule: `display()` first, then save — a chart is
never saved without being shown.

1. Worldwide gross — domestic vs international (dumbbell)
2. Which genres travel — international vs domestic index (diverging bars)
3. Biggest *domestic* films, adjusted for inflation (dumbbell)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import duckdb
from src.ingest import load_config
from chart_templates import lollipop, single_ranked_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: viz notebooks only read, so they run alongside an open kernel.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
img_w, img_h, _ = PRESETS['twitter_landscape']
out = Path(cfg['paths']['outputs_social']); out.mkdir(parents=True, exist_ok=True)
def money(v):
    return f'${v/1e9:.2f}B' if abs(v) >= 1e9 else f'${v/1e6:.0f}M'

## 1. Where the money comes from: overseas
Top 15 by worldwide gross. Gold = domestic (US/Canada), teal = international.

In [ ]:
# US-made films; top 15 by worldwide gross, then ordered by the split (foreign share).
ww = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE),
    top AS (SELECT w.title, w.release_year, w.domestic_gross, w.foreign_gross,
                   100.0*w.foreign_gross/w.worldwide_gross AS foreign_pct
            FROM films_worldwide w JOIN us ON us.title=w.title AND us.release_year=w.release_year
            ORDER BY w.worldwide_gross DESC LIMIT 15)
    SELECT * FROM top ORDER BY foreign_pct DESC''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
ww['split_label'] = ww['foreign_pct'].apply(lambda p: f'{p:.0f}% abroad')
img2 = lollipop(ww, category_col='label', value_col='foreign_gross', value2_col='domestic_gross',
    label_col='split_label', value_fmt=lambda v: v,  # sorted by split; label the foreign share
    title="Hollywood's biggest films make most of their money abroad",
    subtitle='Top 15 U.S.-produced films by worldwide gross, ordered by the split. Gold = home (U.S. & Canada), teal = rest of world; label = share earned abroad.',
    source='Box Office Mojo, Top Lifetime Grosses (Worldwide) + TMDB origin country — as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Rest of world', value2_label='Home (US/Canada)',
    img_width=img_w, img_height=img_h)
display(img2)
img2.save(out / '02_worldwide_domestic_vs_international.png')

## 2. Share of box office earned abroad, by genre
Of everything a genre earned worldwide, what fraction came from outside the
U.S. & Canada (US-made films). Every genre earns most of its money abroad;
sci-fi is simply the lowest. Plain share — no index, no zero-line.

In [ ]:
genre = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE)
    SELECT gl.genre AS category, COUNT(*) n,
      ROUND(100.0*SUM(w.foreign_gross)/(SUM(w.domestic_gross)+SUM(w.foreign_gross)),1) AS value
    FROM films_worldwide w
    JOIN us ON us.title=w.title AND us.release_year=w.release_year
    JOIN film_genres_long gl ON gl.title=w.title AND gl.release_year=w.release_year
    GROUP BY gl.genre HAVING COUNT(*)>=10 ORDER BY value DESC''').df()
genre['pct_label'] = genre['value'].apply(lambda v: f'{v:.0f}%')
img3 = single_ranked_bars(genre, category_col='category', value_col='value',
    total_label_col='pct_label', bar_color='#005F73',
    title='Every blockbuster genre earns most of its money abroad',
    subtitle='Share of worldwide box office earned OUTSIDE the U.S. & Canada, by genre (top U.S.-made films). Even the lowest — sci-fi — takes ~61% overseas.',
    source='Box Office Mojo (Worldwide) + TMDB genres/origin — as of Sep 2026',
    img_width=img_w, img_height=img_h)
display(img3)
img3.save(out / '03_genre_share_earned_abroad.png')

## 3. Biggest *domestic* films, adjusted for inflation
A different question: within the U.S. & Canada, adjusted for ticket-price
inflation. Teal = adjusted (2022 $), gold = nominal (release $).

In [ ]:
dom = con.execute('''SELECT title, adjusted_gross, nominal_gross, release_year
    FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15''').df()
dom['label'] = dom['title'] + '  (' + dom['release_year'].astype(str) + ')'
img1 = lollipop(dom, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='The biggest DOMESTIC films of all time, adjusted for inflation',
    subtitle='U.S. & Canada only. Teal = adjusted to 2022 $, gold = nominal (release $) — the gap is a century of ticket-price inflation.',
    source='Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, adj. to 2022) — as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (2022 $)', value2_label='Nominal (release $)',
    img_width=img_w, img_height=img_h)
display(img1)
img1.save(out / '01_domestic_adjusted_vs_nominal.png')

---
Three charts written to `outputs/social/`. Watermark `@unwelcomedata`, brand
palette, `twitter_landscape` preset.

## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')